# Multi-Regional AeroMAPS - All Regions (Simple)

This notebook demonstrates AeroMAPS ability to run multiple regions sequentially using the new `MultiRegionalProcess`.
We've used this notebbok for an article submitted, which should be used as a complement to reading the notebook and vice-versa.


## Preamble - Creating regions 

We first create a set of regions corresponding to the scope of the polices considered in the article. 

In [ ]:
import json
import shutil
import time
import urllib.request
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from aeromaps import create_multi_regional_process
from aeromaps.utils.functions import create_partitioning

%matplotlib widget

In [ ]:
# ── Continent base palette (used throughout the notebook) ─────────────────
continent_palette = {
    "Africa": "#e41a1c",
    "Asia": "#377eb8",
    "Europe": "#4daf4a",
    "North America": "#984ea3",
    "South America": "#ff7f00",
    "Oceania": "#a65628",
}

# ── Sub-region colors ─────────────────────────────────────────────────────
region_colors = {
    "EU_EEA": "#4daf4a",  # EU-27 + Norway + Switzerland
    "UK": "#1e6e23",  # United Kingdom
    "Other_Europe": "#9ed89e",  # Other Europe (incl. Russia)
    "Eurasia_Mix": "#6aabab",  # Russia + Turkey (blend Other Europe/Other Asia)
    "USA": "#984ea3",  # USA
    "Other_NAM": "#c89fcd",  # Other North America
    "Brazil": "#ff7f00",  # Brazil
    "Other_SAM": "#ffb366",  # Other South America
    "Singapore": "#6baed6",  # Singapore
    "Other_Asia": "#377eb8",  # Other Asia (incl. Turkey)
    "Africa": "#e41a1c",  # Africa
    "Oceania": "#a65628",  # Oceania
    "Other": "#d9d9d9",  # Antarctica / uncategorized
}

_LEGEND_LABELS = {
    "EU_EEA": "EU-27 + Norway + Switzerland",
    "UK": "United Kingdom",
    "Other_Europe": "Other Europe",
    "Eurasia_Mix": "Other Europe/Other Asia (airports locations)",
    "USA": "USA",
    "Other_NAM": "Other North America",
    "Brazil": "Brazil",
    "Other_SAM": "Other South America",
    "Other_Asia": "Other Asia",
    "Singapore": "Singapore",
    "Africa": "Africa",
    "Oceania": "Oceania",
}

# ── Country ISO sets ──────────────────────────────────────────────────────
EU_EEA_ISO = {
    "AUT",
    "BEL",
    "BGR",
    "HRV",
    "CYP",
    "CZE",
    "DNK",
    "EST",
    "FIN",
    "FRA",
    "DEU",
    "GRC",
    "HUN",
    "IRL",
    "ITA",
    "LVA",
    "LTU",
    "LUX",
    "MLT",
    "NLD",
    "POL",
    "PRT",
    "ROU",
    "SVK",
    "SVN",
    "ESP",
    "SWE",
    "NOR",
    "CHE",
}
UK_ISO = {"GBR"}
RUSSIA_ISO = {"RUS"}
TURKEY_ISO = {"TUR"}
USA_ISO = {"USA"}
BRAZIL_ISO = {"BRA"}
SINGAPORE_ISO = {"SGP"}

# ── Download Natural Earth 110m attribute data (ISO + continent) ──────────
SOURCES = [
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson",
    "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.geojson",
]
_geojson = None
for _url in SOURCES:
    try:
        with urllib.request.urlopen(_url, timeout=20) as _resp:
            _geojson = json.loads(_resp.read())
        break
    except Exception:
        pass


def _get_iso(props):
    for k in ("ISO_A3_EH", "ISO_A3", "ADM0_A3"):
        v = (props.get(k) or "").strip()
        if v and v != "-99":
            return v
    return ""


_records = [
    {"iso": _get_iso(f["properties"]), "continent": f["properties"].get("CONTINENT", "")}
    for f in _geojson["features"]
]
_df = pd.DataFrame(_records)
_df = _df[_df["iso"] != ""]


# ── Classify each country into a sub-region ──────────────────────────────
def _classify(iso, cont):
    if iso in EU_EEA_ISO:
        return "EU_EEA"
    if iso in UK_ISO:
        return "UK"
    if iso in RUSSIA_ISO:
        return "Eurasia_Mix"
    if iso in TURKEY_ISO:
        return "Eurasia_Mix"
    if iso in USA_ISO:
        return "USA"
    if iso in BRAZIL_ISO:
        return "Brazil"
    if iso in SINGAPORE_ISO:
        return "Singapore"
    c = (cont or "").strip().lower()
    if c == "europe":
        return "Other_Europe"
    if c == "north america":
        return "Other_NAM"
    if c == "south america":
        return "Other_SAM"
    if c == "asia":
        return "Other_Asia"
    if c == "africa":
        return "Africa"
    if c == "oceania":
        return "Oceania"
    return "Other"


_df["sub_region"] = _df.apply(lambda r: _classify(r["iso"], r["continent"]), axis=1)

# ── Build discrete (step) colorscale ─────────────────────────────────────
_order = list(region_colors.keys())  # 12 entries
_n = len(_order)
_df["z"] = _df["sub_region"].map({r: i for i, r in enumerate(_order)}).fillna(_n - 1).astype(float)

_colorscale = []
for _i, _r in enumerate(_order):
    _colorscale.append([_i / (_n - 1), region_colors[_r]])
    if _i < _n - 1:
        _colorscale.append([(_i + 0.9999) / (_n - 1), region_colors[_r]])

# ── Build figure ──────────────────────────────────────────────────────────
fig = go.Figure()

# Choropleth layer
fig.add_trace(
    go.Choropleth(
        locations=_df["iso"],
        z=_df["z"],
        locationmode="ISO-3",
        colorscale=_colorscale,
        zmin=0,
        zmax=_n - 1,
        showscale=False,
        marker=dict(line=dict(color="white", width=0.5)),
        customdata=_df["sub_region"],
        hovertemplate="<b>%{location}</b><br>%{customdata}<extra></extra>",
        name="",
    )
)

# Singapore: dot + label (too small for 110m polygons)
fig.add_trace(
    go.Scattergeo(
        lon=[103.82],
        lat=[1.35],
        mode="markers+text",
        marker=dict(
            size=9,
            color=region_colors["Singapore"],
            symbol="circle",
            line=dict(width=1.5, color="black"),
        ),
        text=["Singapore"],
        textposition="middle right",
        textfont=dict(size=9, color="black"),
        showlegend=False,
        hoverinfo="text",
    )
)

# Dummy traces for custom legend (one per sub-region, no geometry)
for _sub, _label in _LEGEND_LABELS.items():
    fig.add_trace(
        go.Scattergeo(
            lon=[None],
            lat=[None],
            mode="markers",
            marker=dict(size=12, color=region_colors[_sub], symbol="square"),
            name=_label,
            showlegend=True,
        )
    )

# ── Layout ────────────────────────────────────────────────────────────────
fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=False,
        showland=False,
        showocean=False,
        oceancolor="rgb(248,248,248)",
        showlakes=False,
        bgcolor="white",
        projection_type="natural earth",
        lataxis_range=[-90, 90],
    ),
    legend=dict(
        x=0.0,
        y=0.03,
        xanchor="left",
        yanchor="bottom",
        bgcolor="rgba(255,255,255,0.3)",
        bordercolor="gray",
        borderwidth=0.5,
        font=dict(size=7),
        title=dict(text="Regions Modelled", font=dict(size=10)),
    ),
    paper_bgcolor="white",
    margin=dict(l=0, r=0, t=0, b=0),
    height=520,
)

# Save as PDF (requires: pip install -U kaleido)
fig.write_image("figures/multi_regional_breakdown.pdf", format="pdf", width=800, height=450)
fig.show()

## 0. Prepare Region Data

Generate partitioning files and config.yaml for each region from their CSV data.

In [ ]:
# Reference files from region_default
default_folder = Path("data/region_default")
energy_carriers_file = default_folder / "energy_carriers_data.yaml"

# All region folders (with actual folder names including typos and asterisks)
region_folders = [
    "region_af_dom",
    "region_af_int",
    "region_as_dom",
    "region_bras_dom",
    "region_eu_dom",
    "region_eu_int",
    "region_oc_dom",
    "region_oc_int",
    "region_other_as_int",
    "region_other_eur_dom",
    "region_other_eur_int",
    "region_other_nam_dom",
    "region_other_nam_int",
    "region_other_sam_dom",
    "region_sin_int",
    "region_uk_dom",
    "region_uk_int",
    "region_usa_dom",
    "region_usa_int",
    "region_sam_int",
]

data_path = Path("data")

for folder_name in region_folders:
    folder_path = data_path / folder_name

    if not folder_path.exists():
        print(f"⚠ Folder not found: {folder_name}")
        continue

    csv_file = folder_path / "dataframe_aeromaps.csv"
    if not csv_file.exists():
        print(f"⚠ CSV not found: {csv_file}")
        continue

    # 1. Generate  a classic (single-region) AeroMAPS partitioning from AeroSCOPE CSV, specifiied in the region's folder.
    create_partitioning(file=str(csv_file), path=str(folder_path))

    print(f"✓ Prepared: {folder_name}")

print(f"\n✓ All {len(region_folders)} regions prepared")

## 1. Create Multi-Regional Process

In [ ]:
# Create the multi-regional process in sequential mode (specified in the regionalisation config file, opposed to single MDA chain)
process = create_multi_regional_process(
    configuration_file="regionalisation_all_regions.yaml", disable_execution_statistics=True
)

print("✓ Process created")
print(f"  Mode: {process.execution_mode}")
print(f"  Regions ({len(process.list_regions())}): {process.list_regions()}")


# Process computation, no parallelisation (could be done but no speed increase found)

process.compute(parallel=False)

## 2.5. ATAG Reference - 2026 S1 Scenario

In this article, we want to explore the gap between sector's pledges and actual policies planned. 
Thus we beleive Air Transport Action Group (ATAG) roadmaps are a sound comparison basis.  
Data from their scenario is extracted using Automeris' WebPlotDigitizer. 

In [ ]:
# LTAG points (year_float, value)
years = [
    2014.9164012738854,
    2015.9195859872611,
    2016.8949044585988,
    2017.3964968152868,
    2017.8980891719746,
    2018.316082802548,
    2018.6226114649683,
    2018.845541401274,
    2018.9291401273886,
    2019.0127388535034,
    2019.1520700636943,
    2019.2914012738854,
    2019.4585987261148,
    2019.7093949044588,
    2019.9880573248408,
    2020.4339171974523,
    2020.9355095541403,
    2021.4928343949045,
    2022.0501592356688,
    2022.6074840764331,
    2022.9140127388537,
    2023.4992038216562,
    2024.9482484076434,
    2027.6234076433122,
    2029.3232484076434,
    2031.3017515923568,
    2033.0573248407645,
    2035.2866242038217,
    2037.125796178344,
    2038.90923566879,
    2040.4140127388537,
    2041.361464968153,
    2042.6154458598728,
    2043.9251592356688,
    2045.1234076433122,
    2046.0151273885351,
    2047.0740445859874,
    2048.2444267515925,
    2049.2476114649685,
    2050.0,
]
values = [
    780.0387596899222,
    813.9534883720928,
    862.4031007751937,
    886.627906976744,
    910.8527131782944,
    915.6976744186045,
    915.6976744186045,
    915.6976744186045,
    915.6976744186045,
    915.6976744186045,
    862.4031007751937,
    794.5736434108528,
    726.7441860465115,
    620.1550387596899,
    508.7209302325582,
    547.4806201550387,
    586.2403100775196,
    663.7596899224804,
    746.1240310077517,
    838.1782945736434,
    891.4728682170542,
    920.5426356589146,
    1017.4418604651162,
    1172.4806201550384,
    1259.68992248062,
    1356.5891472868216,
    1438.9534883720928,
    1545.5426356589146,
    1623.0620155038757,
    1715.1162790697672,
    1792.6356589147285,
    1845.9302325581393,
    1918.6046511627906,
    1986.4341085271317,
    2059.108527131783,
    2126.937984496124,
    2189.9224806201546,
    2272.286821705426,
    2330.426356589147,
    2383.7209302325577,
]

LTAG_series = pd.Series(values, index=years, name="LTAG_series")

# Regularize on integer years with linear interpolation
years_int = np.arange(
    int(np.ceil(LTAG_series.index.min())), int(np.floor(LTAG_series.index.max())) + 1
)
LTAG_series_yearly = (
    LTAG_series.sort_index()
    .reindex(LTAG_series.index.union(years_int))
    .interpolate(method="index")
    .reindex(years_int)
)
LTAG_series_yearly.index.name = "year"
LTAG_series_yearly.name = "LTAG_series_yearly"


# Using CORSIA Methodology to include LCA reduction in combustion-only emissions
# All AeroMAPS emissions are LCA, we have to convert using CORSIA factor.
corsia_ratio = 3.83 / 3.16  # typical EF for fossil kerosene (LCA/Combustion)

# Next, we want to have both secnarios start from the same basis, so we compute an correction factor for 2019.
# It should be close to one in theory.
# Multiplicative calibration so that LTAG(2019) = process overall CO2 incl. energy (2019)
target_2019 = (
    process.data["vector_outputs"]["overall:co2_emissions_including_energy"].loc[2019]
    / corsia_ratio
)
ltag_2019 = LTAG_series_yearly.loc[2019]
LTAG_multiplicative_factor = 1.0  # target_2019 / ltag_2019
LTAG_series_yearly_scaled = LTAG_series_yearly * LTAG_multiplicative_factor
LTAG_series_yearly_scaled.name = "LTAG_ref_scaled"

print(f"AeroMAPS vs ATAG 2019: {85 / (target_2019 / ltag_2019):.6f}")
print(
    f"LTAG scaled 2019: {LTAG_series_yearly_scaled.loc[2019]:.6f} | Target 2019: {target_2019:.6f}"
)

LTAG_series_yearly

In [ ]:
# LTAG reference 2: regularize to integer years + calibrate on 2019
ltag2_years = [
    2022.96974522293,
    2024.8367834394905,
    2026.4808917197454,
    2028.125,
    2029.7133757961785,
    2031.5246815286625,
    2033.0294585987263,
    2034.673566878981,
    2035.704617834395,
    2038.3240445859874,
    2039.9960191082803,
    2042.50398089172,
    2043.9808917197454,
    2045.6528662420383,
    2047.408439490446,
    2048.8017515923566,
    2049.9721337579617,
    2050.0,
]
ltag2_values = [
    896.3178294573643,
    983.5271317829456,
    1056.2015503875966,
    1124.0310077519378,
    1187.0155038759688,
    1245.15503875969,
    1293.6046511627906,
    1346.8992248062013,
    1366.2790697674418,
    1443.798449612403,
    1497.0930232558137,
    1584.302325581395,
    1647.286821705426,
    1715.1162790697672,
    1787.7906976744184,
    1860.4651162790697,
    1913.7596899224804,
    1918.6,
]

LTAG_series_2 = pd.Series(ltag2_values, index=ltag2_years, name="LTAG_series_2")

years_int_2 = np.arange(
    int(np.ceil(LTAG_series_2.index.min())), int(np.floor(LTAG_series_2.index.max())) + 1
)
LTAG_series_2_yearly = (
    LTAG_series_2.sort_index()
    .reindex(LTAG_series_2.index.union(years_int_2))
    .interpolate(method="index")
    .reindex(years_int_2)
)
LTAG_series_2_yearly.index.name = "year"
LTAG_series_2_yearly.name = "LTAG_series_2_yearly"

LTAG_series_2_yearly_scaled = LTAG_series_2_yearly * LTAG_multiplicative_factor
LTAG_series_2_yearly_scaled.name = "LTAG_ref_2_scaled"

LTAG_series_2_yearly_scaled.head()


# LTAG reference 3: regularize to integer years + calibrate on 2019
ltag3_years = [
    2023,
    2024,
    2025,
    2026,
    2027,
    2028,
    2029,
    2030,
    2031,
    2032,
    2033,
    2034,
    2035,
    2036,
    2037,
    2038,
    2039,
    2040,
    2041,
    2042,
    2043,
    2044,
    2045,
    2046,
    2047,
    2048,
    2049,
    2050,
]
ltag3_values = [
    895.8689352849383,
    879.8734112599909,
    909.3399251293149,
    926.4981016031657,
    926.5012098379698,
    936.2148417933402,
    949.3633580267451,
    957.1249186598097,
    957.1249308599911,
    957.126052337891,
    945.4945873653428,
    934.203646469638,
    926.0936045559185,
    941.386012517454,
    881.0343944047679,
    812.1982672463873,
    734.5909004402608,
    646.9878765583012,
    583.3624479511473,
    487.22211344836114,
    403.54393795122496,
    325.1035203928527,
    256.017664016455,
    188.46572371929733,
    133.87174643875778,
    90.251984364907,
    54.74030128095865,
    9.689922480620226,
]

LTAG_series_3 = pd.Series(ltag3_values, index=ltag3_years, name="LTAG_series_3")

years_int_3 = np.arange(
    int(np.ceil(LTAG_series_3.index.min())), int(np.floor(LTAG_series_3.index.max())) + 1
)
LTAG_series_3_yearly = (
    LTAG_series_3.sort_index()
    .reindex(LTAG_series_3.index.union(years_int_3))
    .interpolate(method="index")
    .reindex(years_int_3)
)
LTAG_series_3_yearly.index.name = "year"
LTAG_series_3_yearly.name = "LTAG_series_3_yearly"

LTAG_series_3_yearly_scaled = LTAG_series_3_yearly * LTAG_multiplicative_factor
LTAG_series_3_yearly_scaled.name = "LTAG_ref_3_scaled"

LTAG_series_3_yearly_scaled = LTAG_series_3_yearly_scaled.reindex(range(2019, 2051))

LTAG_series_3_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]


# LTAG reference 4 (Future Aircraft): regularize to integer years + calibrate on 2019
ltag4_years = [
    2022.9418789808917,
    2023.7778662420383,
    2025.4498407643314,
    2026.954617834395,
    2028.5987261146497,
    2030.9673566878982,
    2033.0573248407645,
    2034.8686305732485,
    2037.2372611464968,
    2039.187898089172,
    2040.8877388535034,
    2042.531847133758,
    2044.2316878980894,
    2045.875796178344,
    2047.8264331210191,
    2050.0,
]
ltag4_values = [
    896.3178294573643,
    925.3875968992247,
    1007.751937984496,
    1080.426356589147,
    1143.410852713178,
    1230.6201550387595,
    1293.6046511627906,
    1342.0542635658915,
    1400.1937984496124,
    1443.798449612403,
    1472.8682170542634,
    1506.782945736434,
    1540.6976744186045,
    1574.612403100775,
    1623.0620155038757,
    1686.0465116279067,
]

LTAG_series_4 = pd.Series(ltag4_values, index=ltag4_years, name="LTAG_series_4")

years_int_4 = np.arange(
    int(np.ceil(LTAG_series_4.index.min())), int(np.floor(LTAG_series_4.index.max())) + 1
)
LTAG_series_4_yearly = (
    LTAG_series_4.sort_index()
    .reindex(LTAG_series_4.index.union(years_int_4))
    .interpolate(method="index")
    .reindex(years_int_4)
)
LTAG_series_4_yearly.index.name = "year"
LTAG_series_4_yearly.name = "LTAG_series_4_yearly"

LTAG_series_4_yearly_scaled = LTAG_series_4_yearly * LTAG_multiplicative_factor
LTAG_series_4_yearly_scaled.name = "LTAG_ref_4_scaled"

LTAG_series_4_yearly_scaled = LTAG_series_4_yearly_scaled.reindex(range(2019, 2051))
LTAG_series_4_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]


# LTAG reference 5 (Ops): regularize to integer years + calibrate on 2019
ltag5_years = [
    2022.8861464968154,
    2023.6664012738854,
    2025.3383757961785,
    2027.7627388535034,
    2029.3232484076434,
    2030.9952229299365,
    2032.5278662420383,
    2033.9769108280257,
    2035.9832802547771,
    2037.5437898089174,
    2039.8566878980894,
    2041.5286624203823,
    2043.7579617834397,
    2046.2937898089174,
    2048.272292993631,
    2050.0,
]
ltag5_values = [
    896.3178294573643,
    920.5426356589146,
    993.2170542635656,
    1080.426356589147,
    1133.720930232558,
    1182.1705426356586,
    1220.9302325581396,
    1254.84496124031,
    1293.6046511627906,
    1312.984496124031,
    1342.0542635658915,
    1371.124031007752,
    1395.3488372093022,
    1429.2635658914728,
    1472.8682170542634,
    1506.782945736434,
]

LTAG_series_5 = pd.Series(ltag5_values, index=ltag5_years, name="LTAG_series_5")

years_int_5 = np.arange(
    int(np.ceil(LTAG_series_5.index.min())), int(np.floor(LTAG_series_5.index.max())) + 1
)
LTAG_series_5_yearly = (
    LTAG_series_5.sort_index()
    .reindex(LTAG_series_5.index.union(years_int_5))
    .interpolate(method="index")
    .reindex(years_int_5)
)
LTAG_series_5_yearly.index.name = "year"
LTAG_series_5_yearly.name = "LTAG_series_5_yearly"

LTAG_series_5_yearly_scaled = LTAG_series_5_yearly * LTAG_multiplicative_factor
LTAG_series_5_yearly_scaled.name = "LTAG_ref_5_scaled"

LTAG_series_5_yearly_scaled = LTAG_series_5_yearly_scaled.reindex(range(2019, 2051))
LTAG_series_5_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]


# LTAG reference 6 (Energy): regularize to integer years + calibrate on 2019
ltag6_years = [
    2023.0,
    2024.1679936305734,
    2025.5891719745223,
    2027.2054140127389,
    2028.9331210191083,
    2030.7165605095543,
    2032.4164012738854,
    2033.7261146496817,
    2034.9243630573249,
    2035.8996815286625,
    2037.125796178344,
    2038.546974522293,
    2039.940286624204,
    2040.859872611465,
    2041.8351910828026,
    2043.0891719745223,
    2044.454617834395,
    2045.597133757962,
    2046.8232484076434,
    2048.4116242038217,
    2050.0,
]
ltag6_values = [
    901.1627906976744,
    939.9224806201548,
    993.2170542635656,
    1041.6666666666665,
    1080.426356589147,
    1099.8062015503874,
    1094.9612403100773,
    1075.581395348837,
    1041.6666666666665,
    1007.751937984496,
    949.612403100775,
    872.0930232558139,
    794.5736434108528,
    741.2790697674418,
    683.1395348837209,
    605.6201550387595,
    537.7906976744184,
    494.18604651162786,
    455.42635658914696,
    421.5116279069766,
    402.13178294573663,
]

LTAG_series_6 = pd.Series(ltag6_values, index=ltag6_years, name="LTAG_series_6")

years_int_6 = np.arange(
    int(np.ceil(LTAG_series_6.index.min())), int(np.floor(LTAG_series_6.index.max())) + 1
)
LTAG_series_6_yearly = (
    LTAG_series_6.sort_index()
    .reindex(LTAG_series_6.index.union(years_int_6))
    .interpolate(method="index")
    .reindex(years_int_6)
)
LTAG_series_6_yearly.index.name = "year"
LTAG_series_6_yearly.name = "LTAG_series_6_yearly"

LTAG_series_6_yearly_scaled = LTAG_series_6_yearly * LTAG_multiplicative_factor
LTAG_series_6_yearly_scaled.name = "LTAG_ref_6_scaled"

LTAG_series_6_yearly_scaled = LTAG_series_6_yearly_scaled.reindex(range(2019, 2051))
LTAG_series_6_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]

LTAG_series_6_yearly_scaled.head()

## 3. View Outputs

All outputs are stored with namespace prefixes in `data["vector_outputs"]`.

In [ ]:
# Global aggregated outputs
global_outputs = process.get_global_outputs()
print(f"Global outputs shape: {global_outputs.shape}")
global_outputs.head()

In [ ]:
# All outputs with namespaces
print(f"Total columns in vector_outputs: {len(process.data['vector_outputs'].columns)}")
process.data["vector_outputs"].columns.tolist()[:20]  # First 20

In [ ]:
# Regional outputs for a specific region
eu_dom = process.get_regional_outputs("EU_DOM")
print(f"EU_DOM outputs shape: {eu_dom.shape}")
eu_dom.head()

## 3.5. Alternative Scenario: Without New Generation Aircraft

Run a scenario where only fleet renewal with recent aircraft exists (no new generation aircraft like 2035 models).

> **⚠️ Provisional workaround**: This approach uses a temporary swap of the `fleet.yaml` file. 
> An issue is open to enable a cleaner override at the multi-regional configuration level.
> See: https://github.com/AeroMAPS/AeroMAPS/issues/XXX

In [ ]:
from pathlib import Path
from aeromaps import create_multi_regional_process

# Paths
fleet_dir = Path("data/fleet")
fleet_original = fleet_dir / "fleet.yaml"
fleet_no_ng = fleet_dir / "fleet_no_ng.yaml"
fleet_backup = fleet_dir / "fleet_backup.yaml"

# 1. Backup the original fleet.yaml
shutil.copy(fleet_original, fleet_backup)
print("✓ Backed up fleet.yaml → fleet_backup.yaml")

# 2. Replace with fleet_no_ng.yaml (no new generation aircraft)
shutil.copy(fleet_no_ng, fleet_original)
print("✓ Replaced fleet.yaml with fleet_no_ng.yaml")

try:
    # 3. Create and run the process WITHOUT new generation aircraft
    process_no_ng = create_multi_regional_process(
        configuration_file="regionalisation_all_regions.yaml", disable_execution_statistics=True
    )

    start_time = time.time()
    process_no_ng.compute(parallel=False)
    elapsed = time.time() - start_time
    print(f"\n✓ Scénario 'No NG' terminé en {elapsed:.2f} secondes")

finally:
    # 4. Restore original fleet.yaml
    shutil.copy(fleet_backup, fleet_original)
    print("✓ Restored original fleet.yaml")

## 4. Emissions Analysis

Compare CO2 emissions metrics across all regions.

In [ ]:
process.data["vector_outputs"]["overall:co2_emissions_including_aircraft_efficiency"]

In [ ]:
import numpy as np


region_to_continent = {
    "AF_DOM": "Africa",
    "AF_INT": "Africa",
    "AS_DOM": "Asia",
    "OTHER_AS_INT": "Asia",
    "SIN_INT": "Asia",
    "EU_DOM": "Europe",
    "EU_INT": "Europe",
    "UK_DOM": "Europe",
    "UK_INT": "Europe",
    "OTHER_EUR_DOM": "Europe",
    "OTHER_EUR_INT": "Europe",
    "USA_DOM": "North America",
    "USA_INT": "North America",
    "OTHER_NAM_DOM": "North America",
    "OTHER_NAM_INT": "North America",
    "BRAS_DOM": "South America",
    "OTHER_SAM_DOM": "South America",
    "SAM_INT": "South America",
    "OC_DOM": "Oceania",
    "OC_INT": "Oceania",
}

region_to_name = {
    "AF_DOM": "Dom.",
    "AF_INT": "Intl.",
    "AS_DOM": "Dom.",
    "OTHER_AS_INT": "Other Intl.",
    "SIN_INT": "Singapore Intl.",
    "EU_DOM": "EU* Dom.",
    "EU_INT": "EU* Intl.",
    "UK_DOM": "UK* Dom.",
    "UK_INT": "UK* Intl.",
    "OTHER_EUR_DOM": "Other Dom.",
    "OTHER_EUR_INT": "Other Intl.",
    "USA_DOM": "USA Dom.",
    "USA_INT": "USA Intl.",
    "OTHER_NAM_DOM": "Other Dom.",
    "OTHER_NAM_INT": "Other Intl.",
    "BRAS_DOM": "Brazil Dom.",
    "OTHER_SAM_DOM": "Other Dom.",
    "SAM_INT": "Intl.",
    "OC_DOM": "Dom.",
    "OC_INT": "Intl.",
}


# ── Figure 1: Global CO₂ Emissions – Fill-between Waterfall ──
fig1, ax1 = plt.subplots(figsize=(10, 6))

vo = process.data["vector_outputs"]
vo_no_ng = process_no_ng.data["vector_outputs"]
years = vo.index

# --- Retrieve each successive emission stage (Mt) ---
s_bau = (
    vo["overall:co2_emissions_2019technology"] / corsia_ratio
    if "overall:co2_emissions_2019technology" in vo.columns
    else None
)

s_fleet = (
    vo_no_ng["overall:co2_emissions_including_aircraft_efficiency"] / corsia_ratio
    if "overall:co2_emissions_including_aircraft_efficiency" in vo_no_ng.columns
    else None
)

s_future = (
    vo["overall:co2_emissions_including_aircraft_efficiency"] / corsia_ratio
    if "overall:co2_emissions_including_aircraft_efficiency" in vo.columns
    else None
)

s_ops = (
    vo["overall:co2_emissions_including_load_factor"] / corsia_ratio
    if "overall:co2_emissions_including_load_factor" in vo.columns
    else None
)

s_energy = (
    vo["overall:co2_emissions_including_energy"] / corsia_ratio
    if "overall:co2_emissions_including_energy" in vo.columns
    else None
)

s_net = None
if "overall:co2_emissions_including_energy" in vo.columns and "overall:carbon_offset" in vo.columns:
    s_net = (
        vo["overall:co2_emissions_including_energy"] - vo["overall:carbon_offset"]
    ) / corsia_ratio

# --- Fill-between wedges ---
fills = [
    (s_bau, s_fleet, "#FFE082", 0.55, "Fleet Renewal"),  # yellow light
    (s_fleet, s_future, "#FFC107", 0.55, "Future Aircraft"),  # yellow deeper
    (s_future, s_ops, "#FFB74D", 0.55, "Operations"),  # orange
    (s_ops, s_energy, "#66BB6A", 0.50, "Energy"),  # green
    (s_energy, s_net, "#BDBDBD", 0.45, "Offsets"),  # grey
]


ax1.axhline(y=s_bau.loc[2019], color="#454545", linestyle=":", alpha=0.5, label="2019 emissions")

s_bau.plot(ax=ax1, label="Frozen 2019 tech.", color="#FFE082")
s_net.plot(ax=ax1, color="#BDBDBD")


for top, bottom, color, alpha, label in fills:
    if top is not None and bottom is not None:
        ax1.fill_between(
            years,
            top,
            bottom,
            alpha=alpha,
            color=color,
            label=label,
            edgecolor=color,
            linewidth=0.5,
        )


# --- ATAG reference lines (black, thin, different linestyles) ---
if "LTAG_series_yearly_scaled" in globals():
    LTAG_series_yearly_scaled.plot(
        ax=ax1, label="ATAG - Frozen 2024 tech.", linewidth=1.2, color="black", linestyle="-"
    )
if "LTAG_series_2_yearly_scaled" in globals():
    LTAG_series_2_yearly_scaled.plot(
        ax=ax1, label="ATAG - Fleet Renewal", linewidth=1.2, color="black", linestyle=":"
    )
if "LTAG_series_4_yearly_scaled" in globals():
    LTAG_series_4_yearly_scaled.plot(
        ax=ax1,
        label="ATAG - Future Aircraft",
        linewidth=1.2,
        color="black",
        linestyle=(0, (3, 1, 1, 1)),
    )
if "LTAG_series_5_yearly_scaled" in globals():
    LTAG_series_5_yearly_scaled.plot(
        ax=ax1, label="ATAG - Operations", linewidth=1.2, color="black", linestyle=(0, (5, 1, 1, 1))
    )
if "LTAG_series_6_yearly_scaled" in globals():
    LTAG_series_6_yearly_scaled.plot(
        ax=ax1, label="ATAG - Energy", linewidth=1.2, color="black", linestyle="--"
    )
if "LTAG_series_3_yearly_scaled" in globals():
    LTAG_series_3_yearly_scaled.plot(
        ax=ax1, label="ATAG - Offsets", linewidth=1.2, color="black", linestyle="-."
    )

ax1.set_xlabel("Year")
ax1.set_ylabel("$\mathrm{CO}_2$ Emissions (Mt)")
ax1.set_xlim(2019, 2050)
# ax1.set_title("Global CO₂ Emissions — Scenario Decomposition")
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("figures/global_emissions_waterfall.pdf", format="pdf")
plt.show()

In [ ]:
# --- Relative emissions reductions compared to baseline ---
reduction_ops = ((s_bau.loc[2050] - s_ops.loc[2050]) / s_bau.loc[2050] * 100).round(1)
reduction_energy = ((s_bau.loc[2050] - s_energy.loc[2050]) / s_bau.loc[2050] * 100).round(1)
reduction_net = ((s_bau.loc[2050] - s_net.loc[2050]) / s_bau.loc[2050] * 100).round(1)

print(
    f"Emissions reduction in 2050 vs Baseline: {reduction_ops}% Operations, {reduction_energy}% Energy, {reduction_net}% Net"
)

reduction_energy_wrt_19 = (
    (s_energy.loc[2050] - s_energy.loc[2019]) / s_energy.loc[2019] * 100
).round(1)
print(f"Emissions reduction in 2050 vs 2019: {reduction_energy_wrt_19}% Energy")

reduction_net_wrt_19 = ((s_net.loc[2050] - s_net.loc[2019]) / s_net.loc[2019] * 100).round(1)
print(f"Emissions reduction in 2050 vs 2019: {reduction_net_wrt_19}% Net")

In [ ]:
# ── Figure 2: CO₂ Evolution — Overview + Continent subplots (normalized, base 1 = 2019) ──

vo = process.data["vector_outputs"]
regions = process.list_regions()

continent_palette = {
    "Africa": "#e41a1c",
    "Asia": "#377eb8",
    "Europe": "#4daf4a",
    "North America": "#984ea3",
    "South America": "#ff7f00",
    "Oceania": "#a65628",
}

region_ls_cycle = ["--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 2, 1, 2))]
region_markers_cycle = ["o", "s", "^", "D", "v", "P", "X", "*"]
continents_order = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]


# --- Helper: compute net CO₂ for a list of region ids, normalized to base 1 in 2019 ---
def _net_series(region_ids, normalize=True):
    total = None
    for rid in region_ids:
        c1 = f"{rid}:co2_emissions_including_energy"
        c2 = f"{rid}:carbon_offset"
        if c1 in vo.columns and c2 in vo.columns:
            s = vo[c1] - vo[c2]
            total = s if total is None else total + s
    if total is None:
        return None
    if normalize:
        b = total.loc[2019]
        return total / b if b != 0 else None
    return total


# --- Layout: 1 big overview on top, 6 continent subplots below (2 rows × 3 cols) ---
fig2 = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(3, 3, figure=fig2, height_ratios=[2.3, 1, 1], hspace=0.2, wspace=0.20)
ax_main = fig2.add_subplot(gs[0, :])

# ── Main panel: GLOBAL, DOM, INT + all continents ──
# Global
norm_global = _net_series(regions)
if norm_global is not None:
    ax_main.plot(norm_global.index, norm_global, color="black", linewidth=1.5, label="Global")

# Aggregated DOM / INT
dom_regions = [r for r in regions if "DOM" in r]
int_regions = [r for r in regions if "INT" in r]
norm_dom = _net_series(dom_regions)
norm_int = _net_series(int_regions)
if norm_dom is not None:
    ax_main.plot(
        norm_dom.index, norm_dom, color="black", linewidth=1.5, linestyle="--", label="Domestic"
    )
if norm_int is not None:
    ax_main.plot(
        norm_int.index, norm_int, color="black", linewidth=1.5, linestyle=":", label="International"
    )

# Each continent
for continent in continents_order:
    c_color = continent_palette[continent]
    c_regs = [r for r in regions if region_to_continent.get(r) == continent]
    norm_c = _net_series(c_regs)
    if norm_c is not None:
        ax_main.plot(norm_c.index, norm_c, color=c_color, linewidth=1.5, label=continent)

# ATAG ref 3
if "LTAG_series_3_yearly_scaled" in globals():
    ltag3_b = LTAG_series_3_yearly_scaled.loc[2019]
    if ltag3_b != 0:
        ax_main.plot(
            LTAG_series_3_yearly_scaled.index,
            LTAG_series_3_yearly_scaled / ltag3_b,
            color="#686868",
            linewidth=1.5,
            linestyle="-.",
            label="ATAG - All measures",
        )

ax_main.set_ylabel("Relative $\mathrm{CO}_2$ (1 = 2019)")
ax_main.set_title("Regions Overview", fontsize=12)
ax_main.set_xlim(2019, 2050)
ax_main.legend(loc="upper left", fontsize=8, ncol=2)
ax_main.grid(True, alpha=0.3)
ax_main.axhline(y=1, color="gray", linestyle="--", alpha=0.5)

# ── Continent sub-panels ──
for ci, continent in enumerate(continents_order):
    row = 1 + ci // 3
    col = ci % 3
    ax_sub = fig2.add_subplot(gs[row, col])

    c_color = continent_palette[continent]
    c_regs = [r for r in regions if region_to_continent.get(r) == continent]

    # Continent total (thick)
    norm_c = _net_series(c_regs)
    if norm_c is not None:
        ax_sub.plot(norm_c.index, norm_c, color=c_color, linewidth=1.9, label=continent)

    # Individual regions (thin + markers)
    for i, rid in enumerate(c_regs):
        norm_r = _net_series([rid])
        if norm_r is not None:
            ls = region_ls_cycle[i % len(region_ls_cycle)]
            mk = region_markers_cycle[i % len(region_markers_cycle)]
            ax_sub.plot(
                norm_r.index,
                norm_r,
                color=c_color,
                linewidth=1.2,
                linestyle=ls,
                alpha=0.65,
                marker=mk,
                markersize=3,
                markevery=5,
                label=region_to_name.get(rid, rid),
            )

    if "LTAG_series_3_yearly_scaled" in globals():
        ltag3_b = LTAG_series_3_yearly_scaled.loc[2019]
        if ltag3_b != 0:
            ax_sub.plot(
                LTAG_series_3_yearly_scaled.index,
                LTAG_series_3_yearly_scaled / ltag3_b,
                color="#686868",
                linewidth=1.2,
                linestyle="-.",
                label="ATAG",
            )

    ax_sub.set_title(continent, fontsize=10, color=c_color, fontweight="bold")
    ax_sub.legend(fontsize=6, loc="best")
    ax_sub.grid(True, alpha=0.25)
    ax_sub.set_xlim(2019, 2050)
    ax_sub.axhline(y=1, color="gray", linestyle="--", alpha=0.4)
    ax_sub.set_ylabel("Relative $\mathrm{CO}_2$")
    if row == 2:
        ax_sub.set_xlabel("Year")

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
fig2.savefig(
    figures_dir / "co2_evolution_overview_continents.pdf", format="pdf", bbox_inches="tight"
)
plt.show()

In [ ]:
# ── Figure 3: ETS Allowances & CORSIA Credits (by continent) ──
fig3, ax3 = plt.subplots(figsize=(10, 6))

vo = process.data["vector_outputs"]
regions = process.list_regions()

# Palette: color = continent
continent_palette = {
    "Africa": "#e41a1c",
    "Asia": "#377eb8",
    "Europe": "#4daf4a",
    "North America": "#984ea3",
    "South America": "#ff7f00",
    "Oceania": "#a65628",
}
continents_order = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]

# Linestyle = type of measure
ls_corsia = "-"
ls_ets_eu = ":"
ls_ets_uk = "-."
ls_sum = "--"

# Collect lines grouped by continent: { continent: [(handle, label), ...] }
legend_groups = {c: [] for c in continents_order}
legend_totals = []  # for global totals at the end

# --- ETS per region (EU_DOM and UK_DOM separately) ---
ets_ls = {"EU_DOM": ls_ets_eu, "UK_DOM": ls_ets_uk}
ets_labels = {"EU_DOM": "EU ETS", "UK_DOM": "UK ETS"}
ets_total = None
for region_id in ["EU_DOM", "UK_DOM"]:
    col_fossil = f"{region_id}:energy_consumption_kerosene"
    if col_fossil not in vo.columns:
        continue
    fossil_use = vo[col_fossil].loc[2019:]
    net_co2 = (fossil_use * 73.5) / 1e12  # conversion from MJ to gCO2 to MtCO2
    (line,) = ax3.plot(
        net_co2.index,
        net_co2.values,
        linewidth=1.5,
        color=continent_palette["Europe"],
        linestyle=ets_ls[region_id],
    )
    legend_groups["Europe"].append((line, ets_labels[region_id]))
    ets_total = net_co2.copy() if ets_total is None else ets_total + net_co2

# Free allowances annotation
free_allowances = 20  # MtCO2
ax3.axhline(y=free_allowances, color="black", alpha=0.5, linestyle="-", linewidth=1)
ax3.text(
    2019.5,
    free_allowances * 1.02,
    "Max ETS allowances for SAF purchase",
    color="black",
    alpha=0.7,
    fontsize=7,
    ha="left",
    va="bottom",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.6, pad=0.5),
)

# --- CORSIA by continent (INT regions only) ---
int_regions = [r for r in regions if "INT" in r]
corsia_by_continent = {}
corsia_total = None

for continent in continents_order:
    c_color = continent_palette[continent]
    c_int_regs = [r for r in int_regions if region_to_continent.get(r) == continent]
    if not c_int_regs:
        continue

    continent_offset = None
    for rid in c_int_regs:
        col_offset = f"{rid}:carbon_offset"
        if col_offset in vo.columns:
            s = vo[col_offset] / corsia_ratio
            continent_offset = s.copy() if continent_offset is None else continent_offset + s

    if continent_offset is not None:
        (line,) = ax3.plot(
            continent_offset.index,
            continent_offset.values,
            linewidth=1.5,
            color=c_color,
            linestyle=ls_corsia,
        )
        legend_groups[continent].append((line, "CORSIA"))
        corsia_by_continent[continent] = continent_offset
        corsia_total = (
            continent_offset.copy() if corsia_total is None else corsia_total + continent_offset
        )

# Europe CORSIA + ETS sum
if "Europe" in corsia_by_continent and ets_total is not None:
    eu_total = ets_total + corsia_by_continent["Europe"]
    (line,) = ax3.plot(
        eu_total.index,
        eu_total.values,
        linewidth=2,
        color=continent_palette["Europe"],
        linestyle=ls_sum,
    )
    legend_groups["Europe"].append((line, "CORSIA + ETS"))

# Total ETS
if ets_total is not None:
    (line,) = ax3.plot(ets_total.index, ets_total.values, linewidth=2, color="black", linestyle=":")
    legend_totals.append((line, "Total ETS"))

# Total CORSIA
if corsia_total is not None:
    (line,) = ax3.plot(
        corsia_total.index, corsia_total.values, linewidth=2, color="black", linestyle=ls_corsia
    )
    legend_totals.append((line, "Total CORSIA"))

# Global CORSIA + ETS
if corsia_total is not None and ets_total is not None:
    global_sum = corsia_total.add(ets_total, fill_value=0)
    (line,) = ax3.plot(
        global_sum.index, global_sum.values, linewidth=2, color="black", linestyle=ls_sum
    )
    legend_totals.append((line, "Total CORSIA + ETS"))

# ── Build ordered legend: group by continent, then totals ──
handles, labels = [], []
for continent in continents_order:
    if not legend_groups[continent]:
        continue
    for h, lab in legend_groups[continent]:
        handles.append(h)
        labels.append(f"{continent} — {lab}")
for h, lab in legend_totals:
    handles.append(h)
    labels.append(f"{lab}")

ax3.legend(handles, labels, fontsize=8)
ax3.set_xlabel("Year")
ax3.set_xlim(2019, 2050)
ax3.set_ylabel("Millions ETS Allowances / CORSIA Credits (1 allowance ≈ 1 tCO₂)")
ax3.grid(True, alpha=0.3)
ax3.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
fig3.savefig(figures_dir / "ets_corsia_credits_by_continent.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
process.data["vector_outputs"]["overall:load_factor"]

In [ ]:
# LTAG refs 2 & 3 are now plotted directly in the main plot cell above.

In [ ]:
LTAG_series_3_yearly_scaled

### 4.1 Policy Contribution Treemap

Isolate the cumulative CO₂ reduction attributable to each policy lever, per region:
- **Efficiency** (fleet renewal + operations): `cumulative_co2_emissions_2019technology` − `cumulative_co2_emissions_including_load_factor`
- **Low-carbon energy** (SAF, hydrogen, electrification…): `cumulative_co2_emissions_including_load_factor` − `cumulative_co2_emissions`
- **CORSIA Offsets**: global `cumulative_carbon_offset`

Special groupings: USA_DOM + USA_INT → **SAF GC**, EU_DOM + EU_INT → **ReFuelEU**

In [ ]:
# Treemap "Policy Contribution" avec hierarchie cible:
# Niveau 1: Efficiency, Low-carbon energy, CORSIA Offsets
# Niveau 2:
#   - Efficiency -> Fleet Renewal, Future Aircraft, Operations
#   - Low-carbon energy -> regions
#   - CORSIA Offsets -> aucun enfant

import plotly.graph_objects as go

pio.renderers.default = "vscode"

regions = process.list_regions()
vo = process.data["vector_outputs"]
vo_no_ng = process_no_ng.data["vector_outputs"]

# Regroupements de labels regionaux
merge_groups = {
    "USA_DOM": "SAF GC",
    "USA_INT": "SAF GC",
    "EU_DOM": "ReFuelEU",
    "EU_INT": "ReFuelEU",
    "UK_DOM": "UK SAF",
    "BRAS_DOM": "BRA SAF",
    "SIN_INT": "SIN SAF",
}

# 1) Calcul des reductions par region/policy
rows = []
for region_id in regions:
    col_cumul_2019t = f"{region_id}:cumulative_co2_emissions_2019technology"
    col_cumul_renew = f"{region_id}:cumulative_co2_emissions_including_aircraft_efficiency"
    col_cumul_ae = f"{region_id}:cumulative_co2_emissions_including_aircraft_efficiency"
    col_cumul_lf = f"{region_id}:cumulative_co2_emissions_including_load_factor"
    col_cumul = f"{region_id}:cumulative_co2_emissions"
    col_offset = f"{region_id}:cumulative_carbon_offset"

    display_name = merge_groups.get(region_id, region_to_name.get(region_id, region_id))

    # Fleet Renewal
    if col_cumul_2019t in vo.columns and col_cumul_renew in vo_no_ng.columns:
        renew_reduction = (
            vo.loc[2050, col_cumul_2019t] - vo_no_ng.loc[2050, col_cumul_renew]
        ) / corsia_ratio
        if abs(renew_reduction) > 0.01:
            rows.append(
                {"region": display_name, "policy": "Fleet Renewal", "reduction_Gt": renew_reduction}
            )

    # Future Aircraft
    if col_cumul_renew in vo_no_ng.columns and col_cumul_ae in vo.columns:
        ae_reduction = (
            vo_no_ng.loc[2050, col_cumul_renew] - vo.loc[2050, col_cumul_ae]
        ) / corsia_ratio
        if abs(ae_reduction) > 0.01:
            rows.append(
                {"region": display_name, "policy": "Future Aircraft", "reduction_Gt": ae_reduction}
            )

    # Operations
    if col_cumul_ae in vo.columns and col_cumul_lf in vo.columns:
        lf_reduction = (vo.loc[2050, col_cumul_ae] - vo.loc[2050, col_cumul_lf]) / corsia_ratio
        if abs(lf_reduction) > 0.01:
            rows.append(
                {"region": display_name, "policy": "Operations", "reduction_Gt": lf_reduction}
            )

    # Low-carbon energy
    if col_cumul_lf in vo.columns and col_cumul in vo.columns:
        energy_reduction = (vo.loc[2050, col_cumul_lf] - vo.loc[2050, col_cumul]) / corsia_ratio
        if abs(energy_reduction) > 0.01:
            rows.append(
                {
                    "region": display_name,
                    "policy": "Low-carbon energy",
                    "reduction_Gt": energy_reduction,
                }
            )

    # CORSIA Offsets
    if col_offset in vo.columns:
        offset_val = vo.loc[2050, col_offset] / corsia_ratio
        if abs(offset_val) > 0.01:
            rows.append(
                {"region": display_name, "policy": "CORSIA Offsets", "reduction_Gt": offset_val}
            )

df_policy = pd.DataFrame(rows)
df_policy = df_policy.groupby(["region", "policy"], as_index=False)["reduction_Gt"].sum()
df_policy = df_policy[df_policy["reduction_Gt"] > 0].copy()

# 2) Construction de l'arbre manuel (go.Treemap) pour controler la hierarchie
fleet_total = df_policy.loc[df_policy["policy"] == "Fleet Renewal", "reduction_Gt"].sum()
future_total = df_policy.loc[df_policy["policy"] == "Future Aircraft", "reduction_Gt"].sum()
ops_total = df_policy.loc[df_policy["policy"] == "Operations", "reduction_Gt"].sum()
corsia_total = df_policy.loc[df_policy["policy"] == "CORSIA Offsets", "reduction_Gt"].sum()

df_energy = df_policy[df_policy["policy"] == "Low-carbon energy"].copy()

eff_total = fleet_total + future_total + ops_total
lce_total = df_energy["reduction_Gt"].sum()
root_total = eff_total + lce_total + corsia_total

labels = ["All Reductions", "Efficiency", "Low-carbon energy", "CORSIA Offsets"]
parents = ["", "All Reductions", "All Reductions", "All Reductions"]
values = [root_total, eff_total, lce_total, corsia_total]
color_keys = ["All", "Efficiency", "Low-carbon energy", "CORSIA Offsets"]

# Enfants de Efficiency (niveau 2)
for name, val in [
    ("Fleet Renewal", fleet_total),
    ("Future Aircraft", future_total),
    ("Operations", ops_total),
]:
    if val > 0:
        labels.append(name)
        parents.append("Efficiency")
        values.append(val)
        color_keys.append(name)

# Enfants de Low-carbon energy (niveau 2: regions)
for _, r in df_energy.iterrows():
    labels.append(r["region"])
    parents.append("Low-carbon energy")
    values.append(r["reduction_Gt"])
    color_keys.append("Low-carbon energy")

color_map = {
    "Fleet Renewal": "rgba(255, 224, 130, 0.55)",
    "Future Aircraft": "rgba(255, 193, 7, 0.55)",
    "Operations": "rgba(255, 183, 77, 0.55)",
    "Low-carbon energy": "rgba(102, 187, 106, 0.50)",
    "CORSIA Offsets": "rgba(189, 189, 189, 0.45)",
    "Efficiency": "rgba(255, 214, 79, 0.30)",
    "All": "rgba(200, 200, 200, 0.15)",
}
colors = [color_map[k] for k in color_keys]

fig = go.Figure(
    go.Treemap(
        labels=labels,
        parents=parents,
        values=values,
        branchvalues="total",
        marker=dict(colors=colors, cornerradius=8, line=dict(width=1.0, color="rgba(0,0,0,0.9)")),
        texttemplate="<b>%{label}</b><br>%{value:.1f} Gt",
        hovertemplate="<b>%{label}</b><br>Reduction: %{value:.1f} Gt<extra></extra>",
        tiling=dict(pad=4),
        textposition="middle center",
    )
)

fig.update_layout(
    margin=dict(t=0, l=0, r=0, b=0),
    height=450,
    width=800,
    uniformtext=dict(minsize=12, mode="hide"),
)

fig.write_image("figures/treemap_policies.pdf", format="pdf", width=800, height=450)
fig.show()

## 5. Geographic Visualizations


### 5.1. Treemap: 2050 Emissions and Change vs 2019

Treemap encoding:
- **Area** = regional `co2_emissions_including_energy` in 2050
- **Color** = % change from 2019 to 2050
  - Green: decrease
  - Red: increase

In [ ]:
# Stable renderer in VS Code notebooks
pio.renderers.default = "vscode"

# --- Mapping: AeroMAPS region → Continent ---
region_to_continent = {
    "AF_DOM": "Africa",
    "AF_INT": "Africa",
    "AS_DOM": "Asia",
    "OTHER_AS_INT": "Asia",
    "SIN_INT": "Asia",
    "EU_DOM": "Europe",
    "EU_INT": "Europe",
    "UK_DOM": "Europe",
    "UK_INT": "Europe",
    "OTHER_EUR_DOM": "Europe",
    "OTHER_EUR_INT": "Europe",
    "USA_DOM": "North America",
    "USA_INT": "North America",
    "OTHER_NAM_DOM": "North America",
    "OTHER_NAM_INT": "North America",
    "BRAS_DOM": "South America",
    "OTHER_SAM_DOM": "South America",
    "SAM_INT": "South America",
    "OC_DOM": "Oceania",
    "OC_INT": "Oceania",
}


def build_emissions_table(process_data, year):
    rows = []
    regions = process.list_regions()

    for region_id in regions:
        col = f"{region_id}:co2_emissions_including_energy"
        col_offset = f"{region_id}:carbon_offset"
        if (
            col in process_data["vector_outputs"].columns
            and year in process_data["vector_outputs"].index
        ):
            rows.append(
                {
                    "region_id": region_id,
                    "region": region_to_name.get(region_id, region_id),
                    "continent": region_to_continent.get(region_id, "Unknown"),
                    f"co2_{year}": float(
                        process_data["vector_outputs"].loc[year, col]
                        - process_data["vector_outputs"].loc[year, col_offset]
                    )
                    / corsia_ratio,
                }
            )

    return pd.DataFrame(rows)


# Build and merge 2019/2050 data
df_2019 = build_emissions_table(process.data, 2019)
df_2050 = build_emissions_table(process.data, 2050)

df = df_2050.merge(df_2019[["region_id", "co2_2019"]], on=["region_id"], how="left")


df["delta_pct"] = ((df["co2_2050"] - df["co2_2019"]) / df["co2_2019"]) * 100

### Below; kinda weird treemap df building to keep consistent data (weighted emission increase/decrease)

# Filter strictly positive 2050 emissions
df_pos = df[df["co2_2050"] > 0].copy()
if df_pos.empty:
    print("No strictly positive 2050 emissions available for treemap areas.")
else:
    labels = []
    parents = []
    values = []
    colors = []
    customdata = []
    ids = []  # use region_id for uniqueness
    display_labels = []  # visible names

    # World node
    world_id = "World"
    world_co2_2019 = df_pos["co2_2019"].sum()
    world_co2_2050 = df_pos["co2_2050"].sum()
    world_delta = int(np.rint((world_co2_2050 - world_co2_2019) / world_co2_2019 * 100))

    ids.append(world_id)
    labels.append("World")
    parents.append("")
    values.append(world_co2_2050)
    colors.append(world_delta)
    customdata.append([world_co2_2019, world_co2_2050, world_delta])
    display_labels.append("World")

    # Continents and regions
    for continent in df_pos["continent"].unique():
        sub = df_pos[df_pos["continent"] == continent].copy()
        if sub.empty:
            continue

        # Continent node
        cont_id = f"continent_{continent}"
        cont_co2_2019 = sub["co2_2019"].sum()
        cont_co2_2050 = sub["co2_2050"].sum()
        cont_delta = int(np.rint((cont_co2_2050 - cont_co2_2019) / cont_co2_2019 * 100))

        ids.append(cont_id)
        labels.append(continent)
        parents.append(world_id)
        values.append(cont_co2_2050)
        colors.append(cont_delta)
        customdata.append([cont_co2_2019, cont_co2_2050, cont_delta])
        display_labels.append(continent)

        # Region nodes (region_id is unique)
        for _, row in sub.iterrows():
            ids.append(row["region_id"])
            labels.append(row["region"])
            parents.append(cont_id)
            values.append(row["co2_2050"])
            colors.append(int(np.rint(row["delta_pct"])))
            customdata.append([row["co2_2019"], row["co2_2050"], int(np.rint(row["delta_pct"]))])
            display_labels.append(row["region"])

    max_abs_change = max(abs(df_pos["delta_pct"].min()), abs(df_pos["delta_pct"].max()))

    # Treemap
    fig = go.Figure(
        go.Treemap(
            labels=display_labels,
            ids=ids,
            parents=parents,
            values=values,
            branchvalues="total",
            marker=dict(
                colors=colors,
                colorscale=[
                    [0.0, "#1a9850"],
                    [0.5, "#f7f7f7"],
                    [1.0, "#d73027"],
                ],
                cmin=-max_abs_change,
                cmax=max_abs_change,
                line=dict(width=1.2, color="rgba(0,0,0,0.9)"),
                colorbar=dict(title="2050 vs 2019 (%)", tickformat="+d"),
                cornerradius=6,
            ),
            customdata=customdata,
            texttemplate="%{label}<br><b>%{customdata[2]:+d}%</b><br>(%{customdata[1]:.1f} Mt)",
            hovertemplate=(
                "<b>%{label}</b><br>"
                "2019: %{customdata[0]:.1f} Mt<br>"
                "2050: %{customdata[1]:.1f} Mt<br>"
                "Change: %{customdata[2]:+d}%<extra></extra>"
            ),
            textfont=dict(size=14, color="black"),
            textposition="middle center",
            tiling=dict(pad=2),
            opacity=0.72,
        )
    )

# Layout et export PDF
fig.update_layout(
    margin=dict(t=0, l=0, r=0, b=0),
    height=560,
)

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
try:
    fig.write_image(
        str(figures_dir / "treemap_2050_emissions_vs_2019.pdf"),
        format="pdf",
        width=1200,
        height=800,
    )
except Exception as e:
    print(f"⚠ Could not save 2050 treemap PDF: {e}")

fig.show()

In [ ]:
df

### 5.2 Geo Treemaps

In [ ]:
# 5.2: geo-inspired treemap + world map background
# Domain sizes are proportional to 2050 emissions across continents.

import plotly.graph_objects as go
import numpy as np
import json
import urllib.request
import math

if "df" not in globals() or df.empty:
    print("Please run section 5.1 first (it creates 'df').")
else:
    max_abs_change = max(abs(df["delta_pct"].min()), abs(df["delta_pct"].max()))
    print(max_abs_change)

    # Geographic centers for each continent (in [0,1]×[0,1] normalized sæpace)
    continent_centers_halfsides_corr = {
        "North America": (0.25, 0.78, 1.4),
        "South America": (0.31, 0.34, 1.1),
        "Europe": (0.5, 0.78, 1.3),
        "Africa": (0.52, 0.46, 1.2),
        "Asia": (0.88, 0.72, 1.2),
        "Oceania": (0.9, 0.24, 1.5),
    }

    # Compute 2050 emissions per continent (only positive regions)
    df_pos = df[df["co2_2050"] > 0].copy()
    continent_emissions = df_pos.groupby("continent")["co2_2050"].sum().to_dict()

    # Scale domains: area ∝ emissions, using sqrt for side length
    max_emission = max(continent_emissions.values()) if continent_emissions else 1.0
    # Max half-side for the largest continent (controls overall visual scale)
    MAX_HALF_SIDE = 0.29

    continent_domains = {}
    for continent, (cx, cy, corr) in continent_centers_halfsides_corr.items():
        emi = continent_emissions.get(continent, 0.0)
        if emi <= 0:
            continue
        # half-side proportional to sqrt(emissions / max_emission)
        half = MAX_HALF_SIDE * math.sqrt(emi / max_emission)
        # Clamp to [0, 1] bounds
        x0 = max(0.0, cx - half / corr)
        x1 = min(1.0, cx + half / corr)
        y0 = max(0.0, cy - half * corr)
        y1 = min(1.0, cy + half * corr)
        continent_domains[continent] = dict(x=[x0, x1], y=[y0, y1])

    fig_geo_bg = go.Figure()

    fig_geo_bg.update_layout(
        xaxis=dict(range=[0, 1], visible=False),
        yaxis=dict(range=[0, 1], visible=False),
    )

    # --------------------------------------------------
    # World background via GeoJSON (stdlib only, no geopandas)
    # --------------------------------------------------
    GEOJSON_URL = (
        "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
    )
    GEOJSON_URL_FALLBACK = (
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.geojson"
    )

    def fetch_geojson(url):
        with urllib.request.urlopen(url, timeout=15) as resp:
            return json.loads(resp.read().decode("utf-8"))

    try:
        geojson = fetch_geojson(GEOJSON_URL)
        bg_source = "geo-countries (GitHub)"
    except Exception:
        try:
            geojson = fetch_geojson(GEOJSON_URL_FALLBACK)
            bg_source = "Natural Earth (naciscdn)"
        except Exception as e:
            geojson = None
            print(f"⚠ Could not load world GeoJSON: {e}")

    polygons_drawn = 0

    if geojson is not None:
        for feature in geojson.get("features", []):
            geom = feature.get("geometry")
            if geom is None:
                continue
            geom_type = geom.get("type", "")
            coords_list = geom.get("coordinates", [])

            if geom_type == "Polygon":
                polygons = [coords_list[0]]
            elif geom_type == "MultiPolygon":
                polygons = [poly[0] for poly in coords_list]
            else:
                continue

            for ring in polygons:
                try:
                    xs = [float((lon + 180.0) / 360.0) for lon, lat, *_ in ring]
                    ys = [float((lat + 90.0) / 180.0) for lon, lat, *_ in ring]
                    fig_geo_bg.add_trace(
                        go.Scatter(
                            x=xs,
                            y=ys,
                            mode="none",
                            fill="toself",
                            fillcolor="rgba(120,120,120,0.24)",
                            line=dict(width=1),
                            hoverinfo="skip",
                            showlegend=False,
                        )
                    )
                    polygons_drawn += 1
                except Exception:
                    continue

    # Add continent treemaps on top (sized proportionally to emissions)
    for idx, (continent, domain) in enumerate(continent_domains.items()):
        sub = df_pos[df_pos["continent"] == continent].copy()
        if sub.empty:
            continue

        child_values = sub["co2_2050"].tolist()
        parent_value = sum(child_values)

        labels = [continent] + sub["region"].tolist()
        parents = [""] + [continent] * len(sub)
        values = [parent_value] + child_values

        cont_co2_2019 = sub["co2_2019"].sum()
        cont_co2_2050 = sub["co2_2050"].sum()
        cont_delta = (
            int(np.rint((cont_co2_2050 - cont_co2_2019) / cont_co2_2019 * 100))
            if cont_co2_2019 > 0
            else 0
        )
        texts = [f"{cont_co2_2050:.1f} Mt<br>{cont_delta:.1f}%"] + [
            f"{v:.1f} Mt<br>{p:+.1f}%" for v, p in zip(sub["co2_2050"], sub["delta_pct"])
        ]

        colors = [cont_delta] + sub["delta_pct"].tolist()

        fig_geo_bg.add_trace(
            go.Treemap(
                labels=labels,
                parents=parents,
                values=values,
                branchvalues="total",
                domain=domain,
                marker=dict(
                    colors=colors,
                    colorscale=[
                        [0.0, "#1a9850"],
                        [0.5, "#f7f7f7"],
                        [1.0, "#d73027"],
                    ],
                    cmin=-max_abs_change,
                    cmax=max_abs_change,
                    line=dict(width=1.2, color="rgba(0,0,0,0.9)"),
                    colorbar=dict(title="2050 vs 2019 (%)", tickformat="+d") if idx == 0 else None,
                    showscale=True if idx == 0 else False,
                    cornerradius=6,
                ),
                customdata=[[cont_co2_2019, cont_co2_2050, cont_delta]]
                + list(zip(sub["co2_2019"], sub["co2_2050"], sub["delta_pct"])),
                texttemplate="%{label}<br><b>%{customdata[2]:+d}%</b><br>(%{customdata[1]:.1f} Mt)",
                hovertemplate=(
                    "<b>%{label}</b><br>"
                    "2019: %{customdata[0]:.1f} Mt<br>"
                    "2050: %{customdata[1]:.1f} Mt<br>"
                    "Change: %{customdata[2]:+d}%<extra></extra>"
                ),
                textfont=dict(size=14, color="black"),
                textposition="middle center",
                tiling=dict(pad=2),
                opacity=0.72,
            )
        )

        # fix for plotly: Add annotation for specific parent-child pair
        if continent == "Africa":
            dom_row = sub[sub["region"] == "Dom."]
            if not dom_row.empty:
                r = dom_row.iloc[0]

                fig_geo_bg.add_annotation(
                    x=0.567,
                    y=0.39,
                    xref="x",
                    yref="y",
                    text=f"{r['region']}<br><b>{r['delta_pct']:+.1f}%</b><br>({r['co2_2050']:.1f} Mt)",
                    showarrow=True,
                    arrowhead=1,
                    ax=+0,
                    ay=+50,
                    font=dict(size=10, color="black"),
                    align="center",
                    bgcolor=None,
                    opacity=0.72,
                )

        if continent == "Asia":
            singap_row = sub[sub["region"] == "Singapore Intl."]
            if not singap_row.empty:
                r = singap_row.iloc[0]

                fig_geo_bg.add_annotation(
                    x=0.92,
                    y=0.40,
                    xref="x",
                    yref="y",
                    text=f"{r['region']}<br><b>{r['delta_pct']:+.1f}%</b> ({r['co2_2050']:.1f} Mt)",
                    showarrow=True,
                    arrowhead=1,
                    ax=+0,
                    ay=-30,
                    font=dict(size=10, color="black"),
                    align="center",
                    bgcolor=None,
                    opacity=0.72,
                )

        if continent == "Europe":
            uk_row = sub[sub["region"] == "UK* Dom."]
            if not uk_row.empty:
                r = uk_row.iloc[0]

                fig_geo_bg.add_annotation(
                    x=0.586,
                    y=0.62,
                    xref="x",
                    yref="y",
                    text=f"{r['region']}<br><b>{r['delta_pct']:+.1f}%</b><br>({r['co2_2050']:.1f} Mt)",
                    showarrow=True,
                    arrowhead=1,
                    ax=+25,
                    ay=+50,
                    font=dict(size=10, color="black"),
                    align="center",
                    bgcolor=None,
                    opacity=0.72,
                )

            other_eur_row = sub[sub["region"] == "Other Dom."]

            if not other_eur_row.empty:
                r = other_eur_row.iloc[0]

                fig_geo_bg.add_annotation(
                    x=0.535,
                    y=0.654,
                    xref="x",
                    yref="y",
                    text=f"{r['region']}<br><b>{r['delta_pct']:+.1f}%</b><br>({r['co2_2050']:.1f} Mt)",
                    font=dict(size=8, color="black"),
                    showarrow=False,
                    align="center",
                    bgcolor=None,
                    opacity=0.72,
                )

        if continent == "South America":
            bras_row = sub[sub["region"] == "Brazil Dom."]

            if not bras_row.empty:
                r = bras_row.iloc[0]

                fig_geo_bg.add_annotation(
                    x=0.365,
                    y=0.34,
                    xref="x",
                    yref="y",
                    text=f"{r['region']}<br><b>{r['delta_pct']:+.1f}%</b> <br>({r['co2_2050']:.1f} Mt)",
                    font=dict(size=10, color="black"),
                    showarrow=True,
                    arrowhead=1,
                    ax=+60,
                    ay=+0,
                    align="center",
                    bgcolor=None,
                    opacity=0.72,
                )
            other_sam_row = sub[sub["region"] == "Other Dom."]

            if not other_sam_row.empty:
                r = other_sam_row.iloc[0]

                fig_geo_bg.add_annotation(
                    x=0.325,
                    y=0.28,
                    xref="x",
                    yref="y",
                    text=f"{r['region']}<br><b>{r['delta_pct']:+.1f}%</b> ({r['co2_2050']:.1f} Mt)",
                    font=dict(size=10, color="black"),
                    showarrow=True,
                    arrowhead=1,
                    ax=0,
                    ay=+40,
                    align="center",
                    bgcolor=None,
                    opacity=0.72,
                )

    fig_geo_bg.update_layout(
        margin=dict(t=0, l=0, r=0, b=0),
        height=600,
        width=1200,
        uniformtext=dict(minsize=10, mode="hide"),
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
    )

    fig_geo_bg.show()

    fig_geo_bg.write_image(
        "figures/world_evol.pdf",
        format="pdf",
        height=600,
        width=1200,
    )

## 6. Cumulative Emissions Analysis by Continent

Compare cumulative CO₂ emissions (2020–2050) per continent against the LTAG reference budget, using two downscaling methods:
- **Grandfathering**: budget proportional to 2019 emission share
- **Equitable (population)**: budget proportional to average population share (mean 2019/2050)

In [ ]:
# --- 6.1 Cumulative net emissions by continent (2050 value) ---
regions = process.list_regions()

rows = []
for region_id in regions:
    col_cumul = f"{region_id}:cumulative_co2_emissions"
    col_no_energy = f"{region_id}:cumulative_co2_emissions_including_load_factor"
    col_offset = f"{region_id}:cumulative_carbon_offset"
    cumul_co2 = (
        process.data["vector_outputs"].loc[2050, col_cumul] / corsia_ratio
        if col_cumul in process.data["vector_outputs"].columns
        else 0.0
    )
    cumul_co2_no_energy = (
        process.data["vector_outputs"].loc[2050, col_no_energy] / corsia_ratio
        if col_no_energy in process.data["vector_outputs"].columns
        else 0.0
    )
    cumul_off = (
        process.data["vector_outputs"].loc[2050, col_offset] / corsia_ratio
        if col_offset in process.data["vector_outputs"].columns
        else 0.0
    )
    rows.append(
        {
            "region": region_to_name.get(region_id, region_id),
            "continent": region_to_continent.get(region_id, "Unknown"),
            "cumul_co2": cumul_co2,
            "cumul_co2_no_energy": cumul_co2_no_energy,
            "cumul_offset": cumul_off,
            "cumul_net": cumul_co2 - cumul_off,
        }
    )

df_cumul = pd.DataFrame(rows)
df_cumul_continent = df_cumul.groupby("continent")[
    ["cumul_co2", "cumul_co2_no_energy", "cumul_offset", "cumul_net"]
].sum()

print("Cumulative net CO₂ emissions by continent (Gt, 2020-2050):")
print("─" * 60)
for continent in ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]:
    r = df_cumul_continent.loc[continent]
    print(
        f"{continent:15s}: {r['cumul_co2']:8.1f} - {r['cumul_offset']:6.1f} offset = {r['cumul_net']:8.1f} Gt net"
    )

global_cumul_net = df_cumul_continent["cumul_net"].sum()
print(f"\n{'GLOBAL':15s}: {global_cumul_net:.1f} Gt net cumulative")
df_cumul_continent

In [ ]:
# --- 6.2 LTAG cumulative budget vs scenario ---
# Cumulative LTAG emissions from 2020 to 2050 (sum of annual Mt values)
LTAG_cumul_budget = LTAG_series_3_yearly_scaled.loc[2020:2050].sum() / 1000.0  # Convert Mt to Gt
LTAG_cumul_budget_no_offset = LTAG_series_6_yearly_scaled.loc[2020:2050].sum() / 1000.0  # Gt


net_budget_1_5_50 = 500  # GT
carbon_removal_2100 = 527  # GT
aviation_grandfathering = 2.4  # %
gross_carbon_budget = (net_budget_1_5_50 + carbon_removal_2100) * (
    aviation_grandfathering / 100
)  # GT

# Global scenario cumulative (from overall aggregated outputs)
overall_cumul_co2 = (
    process.data["vector_outputs"].loc[2050, "overall:cumulative_co2_emissions"] / corsia_ratio
)
overall_cumul_offset = (
    process.data["vector_outputs"].loc[2050, "overall:cumulative_carbon_offset"] / corsia_ratio
)
scenario_cumul_net = overall_cumul_co2 - overall_cumul_offset

overshoot_pct = (scenario_cumul_net - LTAG_cumul_budget) / LTAG_cumul_budget * 100

print(f"LTAG cumulative budget (2020-2050):  {LTAG_cumul_budget:.1f} Gt")
print(f"PA aligned - aviation grandfathering budget:  {gross_carbon_budget:.1f} Gt")
print(f"Scenario cumulative net (2020-2050): {scenario_cumul_net:.1f} Gt")
print(f"Overshoot: {overshoot_pct:+.1f}%")

In [ ]:
# --- 6.3 Downscaling: Grandfathering vs Equitable (population) ---

# a) Grandfathering: share of 2019 emissions
emissions_2019_by_continent = {}
for region_id in regions:
    col = f"{region_id}:co2_emissions_including_energy"
    if col in process.data["vector_outputs"].columns:
        continent = region_to_continent.get(region_id, "Unknown")
        emissions_2019_by_continent[continent] = (
            emissions_2019_by_continent.get(continent, 0.0)
            + process.data["vector_outputs"].loc[2019, col]
        )

total_emissions_2019 = sum(emissions_2019_by_continent.values())
grandfathering_shares = {
    c: v / total_emissions_2019 for c, v in emissions_2019_by_continent.items()
}

# b) Equitable: share of average population (mean 2019/2050)
population_data = {
    "Africa": {"pop_2019": 1_348_005_692, "pop_2050": 2_466_647_773},
    "Asia": {"pop_2019": 4_651_098_724, "pop_2050": 5_278_869_940},
    "Europe": {"pop_2019": 750_809_114, "pop_2050": 704_554_531},
    "North America": {"pop_2019": 594_346_844, "pop_2050": 687_962_540},
    "Oceania": {"pop_2019": 43_504_534, "pop_2050": 57_688_455},
    "South America": {"pop_2019": 423_548_291, "pop_2050": 468_674_525},
}
pop_avg = {c: (d["pop_2019"] + d["pop_2050"]) / 2 for c, d in population_data.items()}
total_pop_avg = sum(pop_avg.values())
equitable_shares = {c: v / total_pop_avg for c, v in pop_avg.items()}

# Build comparison table
comparison_rows = []
continents_order = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]

for continent in continents_order:
    actual = df_cumul_continent.loc[continent, "cumul_net"]

    budget_gf = LTAG_cumul_budget * grandfathering_shares[continent]
    overshoot_gf = (actual - budget_gf) / budget_gf * 100

    budget_eq = LTAG_cumul_budget * equitable_shares[continent]
    overshoot_eq = (actual - budget_eq) / budget_eq * 100

    comparison_rows.append(
        {
            "Continent": continent,
            "CO2 cumul. net (Gt)": round(actual, 1),
            "GF share (%)": round(grandfathering_shares[continent] * 100, 1),
            "GF budget (Gt)": round(budget_gf, 1),
            "GF overshoot (%)": round(overshoot_gf, 1),
            "Pop share (%)": round(equitable_shares[continent] * 100, 1),
            "Pop budget (Gt)": round(budget_eq, 1),
            "Pop overshoot (%)": round(overshoot_eq, 1),
        }
    )

df_comparison = pd.DataFrame(comparison_rows)
print("Downscaling comparison — Grandfathering vs Equitable (population)")
print("=" * 90)
df_comparison

In [ ]:
# --- 6.4 Visualization: Circular bar plot (Planetary Boundaries style) ---
%matplotlib widget

actual_vals = np.array([df_cumul_continent.loc[c, "cumul_net"] for c in continents_order])
actual_vals_no_offsets = np.array(
    [df_cumul_continent.loc[c, "cumul_co2"] for c in continents_order]
)
gf_vals = np.array([LTAG_cumul_budget * grandfathering_shares[c] for c in continents_order])
eq_vals = np.array([LTAG_cumul_budget * equitable_shares[c] for c in continents_order])

n = len(continents_order)
wedge_angle = 2 * np.pi / n
separator_half_width = 0.005

r_budget = 1.0


r_cap_eq = 4

fig, (ax_gf, ax_eq) = plt.subplots(1, 2, figsize=(12, 7), subplot_kw=dict(projection="polar"))

for ax, budget_vals, title, use_cap in [
    (ax_gf, gf_vals, "2019 emissions grandfathering \n (ATAG S1 deviation)", False),
    (ax_eq, eq_vals, "2019-2050 population-based  \n (ATAG S1 deviation)", True),
]:
    ratios = actual_vals / budget_vals
    ratios_no_offsets = actual_vals_no_offsets / budget_vals
    r_max_actual = ratios_no_offsets.max()

    if use_cap:
        r_outer = r_cap_eq + 0.55
    else:
        r_outer = r_max_actual + 0.55

    for i in range(n):
        theta_start = i * wedge_angle + separator_half_width
        theta_end = (i + 1) * wedge_angle - separator_half_width
        thetas = np.linspace(theta_start, theta_end, 80)

        r_actual = ratios[i]
        r_actual_no_offsets = ratios_no_offsets[i]

        if r_actual <= r_budget:
            ax.fill_between(thetas, 0, r_actual, color="#1a9850", alpha=0.7)
            ax.plot(
                thetas,
                np.full_like(thetas, r_actual_no_offsets, dtype=float),
                color="#1a9850",
                linewidth=1,
                linestyle="--",
                alpha=0.7,
            )
        else:
            ax.fill_between(thetas, 0, r_budget, color="#1a9850", alpha=0.7)
            if not use_cap or r_actual <= r_cap_eq:
                ax.fill_between(thetas, r_budget, r_actual, color="#d73027", alpha=0.65)
                ax.plot(
                    thetas,
                    np.full_like(thetas, r_actual_no_offsets, dtype=float),
                    color="#d73027",
                    linewidth=1,
                    linestyle="--",
                    alpha=0.7,
                )
            else:
                ax.fill_between(
                    thetas, r_budget, r_cap_eq, color="#d73027", alpha=0.65, linewidth=0
                )
                n_fade = 15
                fade_r = np.linspace(r_cap_eq, r_cap_eq + 0.45, n_fade + 1)
                for j in range(n_fade):
                    alpha_fade = 0.45 * (1 - j / n_fade)
                    ax.fill_between(
                        thetas, fade_r[j], fade_r[j + 1], color="#d73027", alpha=alpha_fade
                    )

        # Separator lines
        for theta_boundary in [i * wedge_angle, (i + 1) * wedge_angle]:
            ax.plot(
                [theta_boundary, theta_boundary],
                [0, r_outer - 0.1],
                color="black",
                linewidth=1.2,
                alpha=0.75,
            )

        # Horizontal text label at the midpoint of the wedge
        overshoot_pct = (r_actual - 1) * 100
        mid_theta = (theta_start + theta_end) / 2
        label_text = f"{continents_order[i]}\n({overshoot_pct:+.0f}%)"

        ax.text(
            mid_theta,
            r_outer - 0.08,
            label_text,
            ha="center",
            va="center",
            fontsize=7.5,
            fontweight="bold",
            rotation=0,
        )

    # Budget reference ring
    ring_thetas = np.linspace(0, 2 * np.pi, 300)
    ax.plot(
        ring_thetas,
        np.full_like(ring_thetas, r_budget),
        color="black",
        linewidth=1.8,
        linestyle="--",
        alpha=0.7,
    )

    ax.set_ylim(0, r_outer + 0.7)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.grid(False)
    ax.spines["polar"].set_visible(False)
    ax.set_title(title, fontsize=12)

# Legend
safe_patch = mpatches.Patch(color="#1a9850", alpha=0.7, label="Within budget")
over_patch = mpatches.Patch(color="#d73027", alpha=0.65, label="Beyond budget")
no_ofst_line = plt.Line2D(
    [0], [0], color="#d73027", linewidth=1, linestyle="--", label="Without offsets"
)
budget_line = plt.Line2D(
    [0], [0], color="black", linewidth=1.5, linestyle="--", label="ATAG S1 budget"
)

fig.legend(
    handles=[safe_patch, over_patch, no_ofst_line, budget_line],
    loc="lower center",
    ncol=5,
    fontsize=10,
    frameon=False,
)

# fig.suptitle("Cumulative CO₂ emissions vs ATAG S1 budget by continent (2020–2050)",
#             fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / "cumulative_budget_circular_bars.pdf", format="pdf", bbox_inches="tight")
plt.show()

## 7. Sensitivity Analysis

Two alternative scenarios to test the robustness of the baseline multi-regional results:

1. **7.1 — USA without SAF**: Re-run the multi-regional model with the SAF mandate zeroed out for USA_DOM and USA_INT
2. **7.2 — Global ReFuelEU**: Run a standard (non-regional) AeroMAPS process where the ReFuelEU-inspired SAF mandate applies worldwide

### 7.1 USA without SAF mandate

Neutralise the SAF Grand Challenge mandate for USA_DOM and USA_INT by running a mini multi-regional process (2 regions only) with zeroed-out SAF quantities, then compare with the reference results.

In [ ]:
# Run a mini multi-regional process with only USA_DOM and USA_INT (zero SAF use)
process_usa_no_saf = create_multi_regional_process(
    configuration_file="data/sensitivity_usa_no_saf/regionalisation_usa_no_saf.yaml",
    disable_execution_statistics=True,
)
start_time = time.time()
process_usa_no_saf.compute(parallel=False)
elapsed = time.time() - start_time
print(f"✓ Scenario 'USA no SAF' completed in {elapsed:.2f}s (2 regions only)")

In [ ]:
%matplotlib widget

fig, (ax2, ax1) = plt.subplots(1, 2, figsize=(10, 5))

vo_ref = process.data["vector_outputs"]
vo_nosaf = process_usa_no_saf.data["vector_outputs"]

# Reconstruct "global no-SAF" by replacing USA contributions in the reference
# Global_no_saf = Global_ref - USA_ref + USA_no_saf

usa_ref = (
    vo_ref["USA_DOM:co2_emissions_including_energy"]
    + vo_ref["USA_INT:co2_emissions_including_energy"]
) / corsia_ratio

usa_ref_w_off = (
    vo_ref["USA_DOM:co2_emissions_including_energy"]
    + vo_ref["USA_INT:co2_emissions_including_energy"]
    - vo_ref["USA_INT:carbon_offset"]
    - vo_ref["USA_DOM:carbon_offset"]
) / corsia_ratio

usa_nosaf = (vo_nosaf["overall:co2_emissions_including_energy"]) / corsia_ratio
usa_nosaf_w_off = (
    vo_nosaf["overall:co2_emissions_including_energy"] - vo_nosaf["USA_INT:carbon_offset"]
) / corsia_ratio

ref_global = (vo_ref["overall:co2_emissions_including_energy"]) / corsia_ratio
ref_global_w_off = (
    vo_ref["overall:co2_emissions_including_energy"] - vo_ref["overall:carbon_offset"]
) / corsia_ratio

nosaf_global = ref_global - usa_ref + usa_nosaf
nosaf_global_w_off = ref_global_w_off - usa_ref_w_off + usa_nosaf_w_off


# Cumulative difference
usa_ref_cumul = (
    vo_ref.loc[2050, "USA_DOM:cumulative_co2_emissions"]
    + vo_ref.loc[2050, "USA_INT:cumulative_co2_emissions"]
) / corsia_ratio
usa_nosaf_cumul = vo_nosaf.loc[2050, "overall:cumulative_co2_emissions"] / corsia_ratio
delta = usa_nosaf_cumul - usa_ref_cumul

# Cumulative difference with offsets
usa_ref_cumul_w_off = (
    vo_ref.loc[2050, "USA_DOM:cumulative_co2_emissions"]
    + vo_ref.loc[2050, "USA_INT:cumulative_co2_emissions"]
    - vo_ref.loc[2050, "USA_DOM:cumulative_carbon_offset"]
    - vo_ref.loc[2050, "USA_INT:cumulative_carbon_offset"]
) / corsia_ratio

usa_nosaf_cumul_w_off = (
    vo_nosaf.loc[2050, "overall:cumulative_co2_emissions"]
    - vo_nosaf.loc[2050, "overall:cumulative_carbon_offset"]
) / corsia_ratio


delta_w_off = usa_nosaf_cumul_w_off - usa_ref_cumul_w_off

print(f"\nCumulative CO₂ difference at 2050 (No USA SAF GC vs Reference): {delta:.2f} Gt")
print(f"Cumulative CO₂ difference at 2050 with offsets: {delta_w_off:.2f} Gt")

# --- right: Global emissions comparison ---
ref_global.plot(ax=ax1, linewidth=2, color="#2ca02c", linestyle="--", label="")
ref_global_w_off.plot(ax=ax1, linewidth=2, color="#2ca02c")
nosaf_global.plot(ax=ax1, linewidth=2, color="#d62728", linestyle="--")
nosaf_global_w_off.plot(ax=ax1, linewidth=2, color="#d62728", linestyle="-")

# Fill between scenarios (without offsets / with offsets)
ax1.fill_between(
    ref_global.index,
    ref_global.values,
    nosaf_global.values,
    color="#ff9896",
    alpha=0.20,
)

# add the value of the cumulative difference in 2050 as text
ax1.text(
    2046,
    nosaf_global.loc[2050] - 126,
    f"$\Delta$={delta:.2f} Gt",
    fontsize=9,
    color="#d62728",
    ha="center",
)


ax1.fill_between(
    ref_global_w_off.index,
    ref_global_w_off.values,
    nosaf_global_w_off.values,
    color="#ff9896",
    alpha=0.25,
)

# add the value of the cumulative difference in 2050 as text
ax1.text(
    2046,
    nosaf_global_w_off.loc[2050] - 46,
    f"$\Delta$={delta_w_off:.2f} Gt",
    fontsize=9,
    color="#d62728",
    ha="center",
)

if "LTAG_series_3_yearly_scaled" in globals():
    LTAG_series_3_yearly_scaled.plot(
        ax=ax1, label="ATAG S1", linewidth=2, color="black", linestyle="-."
    )

ax1.set_xlabel("Year")
ax1.set_title("Global $\mathrm{CO}_2$ emissions")
ax1.set_xlim(2019, 2050)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- left: USA only (DOM + INT) ---
usa_ref.plot(ax=ax2, label="Reference policies", linewidth=2, color="#2ca02c", linestyle="--")
usa_ref_w_off.plot(
    ax=ax2,
    label="Reference policies (incl. offsets)",
    linewidth=2.0,
    color="#2ca02c",
    linestyle="-",
)
usa_nosaf.plot(ax=ax2, label="No USA SAF GC", linewidth=2.0, color="#d62728", linestyle="--")
usa_nosaf_w_off.plot(
    ax=ax2, label="No USA SAF GC (incl. offsets)", linewidth=2.5, color="#d62728", linestyle="-"
)

ax2.set_xlabel("Year")
ax2.set_ylabel("$\mathrm{CO}_2$ Emissions (Mt)")
ax2.set_title("USA $\mathrm{CO}_2$ emissions")
ax2.set_xlim(2019, 2050)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)


plt.tight_layout()

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / "sensitivity_usa_no_saf.pdf", format="pdf", bbox_inches="tight")
plt.show()


print(
    f"Abatement reduction in 2050: {(nosaf_global_w_off.loc[2050] - ref_global_w_off.loc[2050]) / ref_global_w_off.loc[2050] * 100:.2f} %"
)

### 7.2 Global ReFuelEU mandate

Run the full multi-regional AeroMAPS process (20 regions), but replace every region's energy carriers with a single ReFuelEU-inspired SAF mandate applied uniformly.

Step mandate (valid until preceding year of next step):
- 2% from 2025, 6% from 2030, 20% from 2035, 34% from 2040, 42% from 2045, 70% in 2050

In [ ]:
# Run the full multi-regional process with ReFuelEU energy carriers for every region
process_refueleu = create_multi_regional_process(
    configuration_file="data/sensitivity_global_refueleu/regionalisation_refueleu.yaml",
    disable_execution_statistics=True,
)

process_refueleu.compute(parallel=False)


process_refueleu_enhanced = create_multi_regional_process(
    configuration_file="data/sensitivity_global_enhanced_refueleu/regionalisation_refueleu.yaml",
    disable_execution_statistics=True,
)

for region in process_refueleu_enhanced.list_regions():
    if "DOM" in region:
        process_refueleu_enhanced.get_regional_process(
            region
        ).parameters.residual_carbon_offset_share_reference_years = [2036, 2037, 2040, 2045, 2050]
        process_refueleu_enhanced.get_regional_process(
            region
        ).parameters.residual_carbon_offset_share_reference_years_values = [7, 7, 20, 50, 100]
    elif "INT" in region:
        process_refueleu_enhanced.get_regional_process(
            region
        ).parameters.carbon_offset_baseline_share_total_emissions_reference_periods = [
            2020,
            2027,
            2036,
            2050,
        ]
        process_refueleu_enhanced.get_regional_process(
            region
        ).parameters.carbon_offset_baseline_share_total_emissions_reference_periods_values = [
            60.0,
            85.0,
            0.0,
        ]
        process_refueleu_enhanced.get_regional_process(
            region
        ).parameters.residual_carbon_offset_share_reference_years = [2036, 2037, 2040, 2045, 2050]
        process_refueleu_enhanced.get_regional_process(
            region
        ).parameters.residual_carbon_offset_share_reference_years_values = [7, 7, 20, 50, 100]
    else:
        print(
            f"⚠️ Region '{region}' does not match expected DOM/INT pattern, skipping parameter adjustments."
        )

process_refueleu_enhanced.compute(parallel=False)

In [ ]:
%matplotlib widget

fig, ax1 = plt.subplots(figsize=(9, 5))

vo_ref = process.data["vector_outputs"]
vo_rfeu = process_refueleu.data["vector_outputs"]
vo_rfeu_enhanced = process_refueleu_enhanced.data["vector_outputs"]

# --- Left: Annual CO2 emissions ---
ref_global = vo_ref["overall:co2_emissions_including_energy"] / corsia_ratio
ref_global.plot(ax=ax1, label="Reference policies", linestyle="--", linewidth=1.5, color="#2ca02c")

ref_global_w_off = (
    vo_ref["overall:co2_emissions_including_energy"] - vo_ref["overall:carbon_offset"]
) / corsia_ratio
ref_global_w_off.plot(
    ax=ax1,
    label="Reference policies (incl. offsets)",
    linewidth=1.5,
    linestyle="-",
    color="#2ca02c",
)

rfeu_global = vo_rfeu["overall:co2_emissions_including_energy"] / corsia_ratio
rfeu_global.plot(ax=ax1, label="Global ReFuelEU", linestyle="--", linewidth=1.5, color="#9467bd")

rfeu_global_w_off = (
    vo_rfeu["overall:co2_emissions_including_energy"] - vo_rfeu["overall:carbon_offset"]
) / corsia_ratio
rfeu_global_w_off.plot(
    ax=ax1, label="Global ReFuelEU (incl. offsets)", linewidth=1.5, linestyle="-", color="#9467bd"
)

rfeu_global_enhanced = vo_rfeu_enhanced["overall:co2_emissions_including_energy"] / corsia_ratio
rfeu_global_enhanced.plot(
    ax=ax1, label="Mandate suggestion", linestyle="--", linewidth=1.5, color="#8c564b"
)

rfeu_global_enhanced_w_off = (
    vo_rfeu_enhanced["overall:co2_emissions_including_energy"]
    - vo_rfeu_enhanced["overall:carbon_offset"]
) / corsia_ratio
rfeu_global_enhanced_w_off.plot(
    ax=ax1,
    label="Mandate suggestion (incl. offsets)",
    linewidth=1.5,
    linestyle="-",
    color="#8c564b",
)

if "LTAG_series_6_yearly_scaled" in globals():
    LTAG_series_6_yearly_scaled.plot(
        ax=ax1, label="ATAG S1", linewidth=1.5, color="black", linestyle="--"
    )

if "LTAG_series_3_yearly_scaled" in globals():
    LTAG_series_3_yearly_scaled.plot(
        ax=ax1, label="ATAG S1 (incl. offsets)", linewidth=1.5, color="black", linestyle="-."
    )


ax1.set_xlabel("Year")
ax1.set_xlim(2019, 2050)
ax1.set_ylabel("CO₂ Emissions (Mt)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)


# cumulative values for 2050 with offsets
ref_cumul = (
    vo_ref["overall:cumulative_co2_emissions"] - vo_ref["overall:cumulative_carbon_offset"]
) / corsia_ratio
rfeu_cumul = (
    vo_rfeu["overall:cumulative_co2_emissions"] - vo_rfeu["overall:cumulative_carbon_offset"]
) / corsia_ratio
rfeu_enhanced_cumul = (
    vo_rfeu_enhanced["overall:cumulative_co2_emissions"]
    - vo_rfeu_enhanced["overall:cumulative_carbon_offset"]
) / corsia_ratio

total_offset_ref = vo_ref["overall:cumulative_carbon_offset"].iloc[-1] / corsia_ratio
total_offset_rfeu = vo_rfeu["overall:cumulative_carbon_offset"].iloc[-1] / corsia_ratio
total_offset_rfeu_enhanced = (
    vo_rfeu_enhanced["overall:cumulative_carbon_offset"].iloc[-1] / corsia_ratio
)


# Add cumulative summary directly on the left chart
summary_txt = (
    r"$\bf{Cumulative\ CO_2\ in\ 2050}$"
    "\n"
    f"Ref: {ref_cumul.iloc[-1]:.1f} Gt ({total_offset_ref:.1f} Gt offset)\n"
    f"Global ReFuelEU: {rfeu_cumul.iloc[-1]:.1f} Gt ({total_offset_rfeu:.1f} Gt offset)\n"
    f"Mandate suggestion: {rfeu_enhanced_cumul.iloc[-1]:.1f} Gt ({total_offset_rfeu_enhanced:.1f} Gt offset)\n"
    f"ATAG S1: {LTAG_cumul_budget:.1f} Gt ({(LTAG_cumul_budget_no_offset - LTAG_cumul_budget):.1f} Gt offset)"
)
ax1.text(
    0.55,
    0.04,
    summary_txt,
    transform=ax1.transAxes,
    va="bottom",
    ha="center",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85, edgecolor="0.8"),
)


plt.tight_layout()

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / "sensitivity_global_refueleu.pdf", format="pdf", bbox_inches="tight")

Biommass / electricity consumption

In [ ]:
vo_rfeu_enhanced["overall:carbon_offset"]

In [ ]:
vo_ref["overall:biomass_total_consumption"]

## Appendix - Regional CO2 emissions scenarios 

In [ ]:
# now we explore each region individually


regions

In [ ]:
process.get_regional_process("AF_INT").plot(
    "air_transport_co2_emissions", save=True, size_inches=(8, 6), remove_title=True
)

In [ ]:
# Appendix: regional waterfall decomposition grouped by continent
import matplotlib.patches as mpatches

vo = process.data["vector_outputs"]
vo_no_ng = process_no_ng.data["vector_outputs"]
years = vo.index
regions = process.list_regions()

region_to_continent_local = {
    "AF_DOM": "Africa",
    "AF_INT": "Africa",
    "AS_DOM": "Asia",
    "OTHER_AS_INT": "Asia",
    "SIN_INT": "Asia",
    "EU_DOM": "Europe",
    "EU_INT": "Europe",
    "UK_DOM": "Europe",
    "UK_INT": "Europe",
    "OTHER_EUR_DOM": "Europe",
    "OTHER_EUR_INT": "Europe",
    "USA_DOM": "N. America",
    "USA_INT": "N. America",
    "OTHER_NAM_DOM": "N. America",
    "OTHER_NAM_INT": "N. America",
    "BRAS_DOM": "S. America",
    "OTHER_SAM_DOM": "S. America",
    "SAM_INT": "S. America",
    "OC_DOM": "Oceania",
    "OC_INT": "Oceania",
}
continents_order = ["Africa", "Asia", "Europe", "N. America", "S. America", "Oceania"]


def _series_or_none(df, col):
    return (df[col] / corsia_ratio) if col in df.columns else None


legend_patches = [
    mpatches.Patch(facecolor="#FFE082", edgecolor="#FFE082", alpha=0.55, label="Fleet Renewal"),
    mpatches.Patch(facecolor="#FFC107", edgecolor="#FFC107", alpha=0.55, label="Future Aircraft"),
    mpatches.Patch(facecolor="#FFB74D", edgecolor="#FFB74D", alpha=0.55, label="Operations"),
    mpatches.Patch(facecolor="#66BB6A", edgecolor="#66BB6A", alpha=0.50, label="Low Carbon Energy"),
    mpatches.Patch(facecolor="#BDBDBD", edgecolor="#BDBDBD", alpha=0.45, label="CORSIA Offsets"),
]

ncols = 4

# Flat ordered list: (continent, region_id)
ordered_regions = []
for continent in continents_order:
    for r in regions:
        if region_to_continent_local.get(r) == continent:
            ordered_regions.append((continent, r))

n_total = len(ordered_regions)
nrows = int(np.ceil(n_total / ncols))


fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(6.30, 6.57),
    sharex=True,
    constrained_layout=False,
)
axes = np.atleast_1d(axes).flatten()

for i, (continent, rid) in enumerate(ordered_regions):
    ax = axes[i]

    s_bau = _series_or_none(vo, f"{rid}:co2_emissions_2019technology")
    s_fleet = _series_or_none(vo_no_ng, f"{rid}:co2_emissions_including_aircraft_efficiency")
    s_future = _series_or_none(vo, f"{rid}:co2_emissions_including_aircraft_efficiency")
    s_ops = _series_or_none(vo, f"{rid}:co2_emissions_including_load_factor")
    s_energy = _series_or_none(vo, f"{rid}:co2_emissions_including_energy")

    s_net = None
    col_energy = f"{rid}:co2_emissions_including_energy"
    col_offset = f"{rid}:carbon_offset"
    if col_energy in vo.columns and col_offset in vo.columns:
        s_net = (vo[col_energy] - vo[col_offset]) / corsia_ratio

    fills = [
        (s_bau, s_fleet, "#FFE082", 0.55),
        (s_fleet, s_future, "#FFC107", 0.55),
        (s_future, s_ops, "#FFB74D", 0.55),
        (s_ops, s_energy, "#66BB6A", 0.50),
        (s_energy, s_net, "#BDBDBD", 0.45),
    ]

    for top, bottom, color, alpha in fills:
        if top is not None and bottom is not None:
            ax.fill_between(
                years, top, bottom, color=color, alpha=alpha, edgecolor=color, linewidth=0.3
            )

    if s_bau is not None:
        ax.plot(years, s_bau, color="#E0A800", linewidth=1.0, alpha=0.6)
    if s_net is not None:
        ax.plot(years, s_net, color="#757575", linewidth=1.0, alpha=0.8)

    region_name = region_to_name.get(rid, rid) if "region_to_name" in globals() else rid
    ax.set_title(f"{continent} — {region_name}", fontsize=6.5, pad=2)
    ax.set_xlim(2019, 2050)
    ax.tick_params(axis="both", labelsize=5.5)
    ax.xaxis.set_major_locator(plt.MultipleLocator(10))
    ax.grid(True, alpha=0.2)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5, linewidth=0.6)

for ax in axes[n_total:]:
    ax.set_visible(False)

fig.supxlabel("Year", fontsize=8, y=0.03)
fig.supylabel("CO₂ Emissions (Mt)", fontsize=8, x=0.04)

fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=5,
    fontsize=6.5,
    bbox_to_anchor=(0.53, 0.0),
    bbox_transform=fig.transFigure,
    frameon=False,
)

fig.tight_layout(w_pad=0.3, h_pad=0.5)


figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
out_path = figures_dir / "regional_emissions_waterfall_all.pdf"
fig.savefig(out_path, format="pdf", bbox_inches="tight")
plt.show()

print(f"Saved: {out_path.name}")